In [ ]:
!pip install -q transformers==5.16.1 datasets==5.0.1 accelerate==1.14.0 peft==0.20.0 bitsandbytes==0.50.2 trl==0.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 27.8 MB/s eta 0:00:00


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/OSTutorLLM"

TRAIN_FILE = os.path.join(PROJECT_DIR, "ostutorllm_train_270.json")
TEST_FILE = os.path.join(PROJECT_DIR, "ostutorllm_test_30.json")

print("Training file exists:", os.path.exists(TRAIN_FILE))
print("Test file exists:", os.path.exists(TEST_FILE))

print("\nTraining file:", TRAIN_FILE)
print("Test file:", TEST_FILE)

Training file exists: False
Test file exists: False

Training file: /content/drive/MyDrive/OSTutorLLM/ostutorllm_train_270.json
Test file: /content/drive/MyDrive/OSTutorLLM/ostutorllm_test_30.json


In [ ]:
import os

print("Files inside MyDrive/OSTutorLLM:")

folder = "/content/drive/MyDrive/OSTutorLLM"

if os.path.exists(folder):
    print(os.listdir(folder))
else:
    print("OSTutorLLM folder does not exist")

Files inside MyDrive/OSTutorLLM:
['ostutor_dataset', 'ostutor_qwen3_8b_test', 'os_dataset_50', 'ostutor_qwen3_8b_final']


In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded:
    print(filename)

Saving ostutorllm_test_30.json to ostutorllm_test_30.json
Saving ostutorllm_train_270.json to ostutorllm_train_270.json

Uploaded files:
ostutorllm_test_30.json
ostutorllm_train_270.json


In [ ]:
import shutil
import os

PROJECT_DIR = "/content/drive/MyDrive/OSTutorLLM"

shutil.copy(
    "/content/ostutorllm_train_270.json",
    os.path.join(PROJECT_DIR, "ostutorllm_train_270.json")
)

shutil.copy(
    "/content/ostutorllm_test_30.json",
    os.path.join(PROJECT_DIR, "ostutorllm_test_30.json")
)

print("Datasets copied to Google Drive successfully!")

print("\nFiles now in OSTutorLLM:")
print(os.listdir(PROJECT_DIR))

Datasets copied to Google Drive successfully!

Files now in OSTutorLLM:
['ostutor_dataset', 'ostutor_qwen3_8b_test', 'os_dataset_50', 'ostutor_qwen3_8b_final', 'ostutorllm_train_270.json', 'ostutorllm_test_30.json']


In [ ]:
import json

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Training examples:", len(train_data))
print("Test examples:", len(test_data))

print("\nFirst training example:")
print(train_data[0])

print("\nFirst test example:")
print(test_data[0])

Training examples: 270
Test examples: 30

First training example:
{'instruction': 'Explain process management in operating systems.', 'input': 'Explain it to a beginner.', 'output': "Process management is the operating system's job of creating, scheduling, and controlling processes. A process is a program that is currently running. The OS gives processes CPU time and manages their resources.", 'id': 1, 'split': 'train'}

First test example:
{'instruction': 'What is a shell in Linux?', 'input': 'Explain simply.', 'output': 'A shell is a command interpreter that accepts commands from a user and starts programs or performs system operations.', 'id': 271, 'split': 'test'}


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)

def convert_to_chat(example):
    user_message = example["instruction"]

    if example["input"]:
        user_message += "\n" + example["input"]

    return {
        "messages": [
            {
                "role": "user",
                "content": user_message
            },
            {
                "role": "assistant",
                "content": example["output"]
            }
        ]
    }

train_dataset = train_dataset.map(convert_to_chat)

print("Training dataset size:", len(train_dataset))
print("\nExample after conversion:")
print(train_dataset[0]["messages"])

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Training dataset size: 270

Example after conversion:
[{'content': 'Explain process management in operating systems.\nExplain it to a beginner.', 'role': 'user'}, {'content': "Process management is the operating system's job of creating, scheduling, and controlling processes. A process is a program that is currently running. The OS gives processes CPU time and manages their resources.", 'role': 'assistant'}]


In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print("Model:", model_name)
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Tokenizer loaded successfully!
Model: Qwen/Qwen3-8B
Vocabulary size: 151643


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Qwen3-8B loaded successfully!")

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-8B loaded successfully!


In [ ]:
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_dataset = train_dataset.map(format_chat)

print("Chat formatting completed!")
print("\nFormatted example:")
print(train_dataset[0]["text"][:1000])

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Chat formatting completed!

Formatted example:
<|im_start|>user
Explain process management in operating systems.
Explain it to a beginner.<|im_end|>
<|im_start|>assistant
<think>

</think>

Process management is the operating system's job of creating, scheduling, and controlling processes. A process is a program that is currently running. The OS gives processes CPU time and manages their resources.<|im_end|>



In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 15,335,424 || all params: 8,206,070,784 || trainable%: 0.1869


In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="/content/ostutorllm_300_training",

    num_train_epochs=3,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=1,
    save_strategy="epoch",

    fp16=False,
    bf16=False,

    max_seq_length=512,

    report_to="none"
)

print("SFT training configuration created successfully!")

SFT training configuration created successfully!


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

print("SFTTrainer created successfully!")

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

SFTTrainer created successfully!


In [ ]:
print("Starting OSTutorLLM fine-tuning...")
print("Training examples:", len(train_dataset))
print("Epochs:", training_args.num_train_epochs)

trainer.train()

print("\nTraining completed successfully!")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting OSTutorLLM fine-tuning...
Training examples: 270
Epochs: 3


Step,Training Loss
1,5.960207
2,5.498860
3,4.804759
4,3.709179
5,3.213835
6,2.675255
7,2.886250
8,2.271246
9,2.105153
10,2.072323



Training completed successfully!


In [ ]:
FINAL_ADAPTER_PATH = "/content/drive/MyDrive/OSTutorLLM/ostutor_qwen3_8b_300_final"

trainer.save_model(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)

print("Final 300-example LoRA adapter saved successfully!")
print("Saved to:", FINAL_ADAPTER_PATH)

Final 300-example LoRA adapter saved successfully!
Saved to: /content/drive/MyDrive/OSTutorLLM/ostutor_qwen3_8b_300_final


In [ ]:
import os

print("Checking saved adapter...\n")

if os.path.exists(FINAL_ADAPTER_PATH):
    print("✅ Adapter folder exists!")
    print("\nFiles saved:")

    for file in os.listdir(FINAL_ADAPTER_PATH):
        print(" -", file)
else:
    print("❌ Adapter folder was not found!")

Checking saved adapter...

✅ Adapter folder exists!

Files saved:
 - README.md
 - adapter_model.safetensors
 - adapter_config.json
 - chat_template.jinja
 - tokenizer_config.json
 - tokenizer.json
 - training_args.bin


In [ ]:
import json

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Held-out test examples:", len(test_data))

print("\nFirst test question:")
print(test_data[0]["instruction"])

print("\nExpected answer:")
print(test_data[0]["output"])

Held-out test examples: 30

First test question:
What is a shell in Linux?

Expected answer:
A shell is a command interpreter that accepts commands from a user and starts programs or performs system operations.


In [ ]:
import torch

model.eval()

question = test_data[0]["instruction"]

messages = [
    {
        "role": "user",
        "content": question
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("QUESTION:")
print(question)

print("\nMODEL ANSWER:")
print(answer)

QUESTION:
What is a shell in Linux?

MODEL ANSWER:
<think>

</think>

A shell is a user interface that accepts commands and passes them to the operating system for execution. Common shells include bash, zsh, and sh.


In [ ]:
import re
import torch

model.eval()

results = []

for i, item in enumerate(test_data):
    question = item["instruction"]
    expected = item["output"]

    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    results.append({
        "id": item.get("id", i + 1),
        "question": question,
        "expected": expected,
        "answer": answer
    })

    print(f"Completed {i+1}/30")

print("\n✅ Evaluation generation completed!")
print("Total test questions:", len(results))

Completed 1/30
Completed 2/30
Completed 3/30
Completed 4/30
Completed 5/30
Completed 6/30
Completed 7/30
Completed 8/30
Completed 9/30
Completed 10/30
Completed 11/30
Completed 12/30
Completed 13/30
Completed 14/30
Completed 15/30
Completed 16/30
Completed 17/30
Completed 18/30
Completed 19/30
Completed 20/30
Completed 21/30
Completed 22/30
Completed 23/30
Completed 24/30
Completed 25/30
Completed 26/30
Completed 27/30
Completed 28/30
Completed 29/30
Completed 30/30

✅ Evaluation generation completed!
Total test questions: 30


In [ ]:
for i, result in enumerate(results):
    print("=" * 80)
    print(f"QUESTION {i+1}")
    print("=" * 80)

    print("Q:", result["question"])
    print("\nEXPECTED:")
    print(result["expected"])

    print("\nMODEL ANSWER:")
    print(result["answer"])

    print(f"\nKeyword coverage: {result['concept_coverage']:.2f}%")

QUESTION 1
Q: What is a shell in Linux?

EXPECTED:
A shell is a command interpreter that accepts commands from a user and starts programs or performs system operations.

MODEL ANSWER:
<think>

</think>

A shell is a user interface that accepts commands and passes them to the operating system for execution. Common shells include bash, zsh, and sh.

Keyword coverage: 52.94%
QUESTION 2
Q: What does the Linux command pwd show?

EXPECTED:
The pwd command prints the current working directory.

MODEL ANSWER:
<think>

</think>

pwd stands for "print working directory" and prints the current directory path.

Keyword coverage: 85.71%
QUESTION 3
Q: What does the Linux command ls do?

EXPECTED:
The ls command lists files and directories in a directory.

MODEL ANSWER:
<think>

</think>

The ls command lists the contents of a directory.

Keyword coverage: 60.00%
QUESTION 4
Q: What does the Linux command cd do?

EXPECTED:
The cd command changes the current working directory.

MODEL ANSWER:
<think>

<

In [ ]:
import pandas as pd

evaluation_df = pd.DataFrame([
    {
        "Question": r["question"],
        "Expected Answer": r["expected"],
        "Model Answer": r["answer"],
        "Keyword Coverage (%)": round(r["concept_coverage"], 2)
    }
    for r in results
])

print("Evaluation table created!")
print("Number of test questions:", len(evaluation_df))

display(evaluation_df)

Evaluation table created!
Number of test questions: 30


,Question,Expected Answer,Model Answer,Keyword Coverage (%)
0,What is a shell in Linux?,A shell is a command interpreter that accepts ...,<think>\n\n</think>\n\nA shell is a user inter...,52.94
1,What does the Linux command pwd show?,The pwd command prints the current working dir...,"<think>\n\n</think>\n\npwd stands for ""print w...",85.71
2,What does the Linux command ls do?,The ls command lists files and directories in ...,<think>\n\n</think>\n\nThe ls command lists th...,60.00
3,What does the Linux command cd do?,The cd command changes the current working dir...,<think>\n\n</think>\n\nThe cd command changes ...,100.00
4,What does chmod do in Linux?,chmod changes the permission bits of a file or...,<think>\n\n</think>\n\nchmod changes the permi...,73.68
5,What does ps show in Linux?,The ps command displays information about proc...,<think>\n\n</think>\n\nps shows information ab...,50.00
6,What does kill do in Linux?,The kill command sends a signal to a process i...,<think>\n\n</think>\n\nkill sends a signal to ...,38.10
7,What is fork followed by exec used for?,A process can use fork to create a child and t...,<think>\n\n</think>\n\nFork creates a child pr...,52.00
8,What is a shell pipeline?,A shell pipeline connects the output of one co...,<think>\n\n</think>\n\nA shell pipeline connec...,83.33
9,What is the root directory in Linux?,The root directory is the top of the Linux fil...,<think>\n\n</think>\n\nThe root directory is t...,92.31


In [ ]:
EVAL_FILE = "/content/drive/MyDrive/OSTutorLLM/ostutorllm_30_test_evaluation.csv"

evaluation_df.to_csv(EVAL_FILE, index=False)

print("Evaluation results saved successfully!")
print("File:", EVAL_FILE)
print("Rows:", len(evaluation_df))

Evaluation results saved successfully!
File: /content/drive/MyDrive/OSTutorLLM/ostutorllm_30_test_evaluation.csv
Rows: 30


In [ ]:
evaluation_df["Semantic Score (0-2)"] = None
evaluation_df["Reason"] = ""

print("Semantic scoring columns added.")
display(
    evaluation_df[
        ["Question", "Model Answer", "Semantic Score (0-2)", "Reason"]
    ]
)

Semantic scoring columns added.


,Question,Model Answer,Semantic Score (0-2),Reason
0,What is a shell in Linux?,<think>\n\n</think>\n\nA shell is a user inter...,None,
1,What does the Linux command pwd show?,"<think>\n\n</think>\n\npwd stands for ""print w...",None,
2,What does the Linux command ls do?,<think>\n\n</think>\n\nThe ls command lists th...,None,
3,What does the Linux command cd do?,<think>\n\n</think>\n\nThe cd command changes ...,None,
4,What does chmod do in Linux?,<think>\n\n</think>\n\nchmod changes the permi...,None,
5,What does ps show in Linux?,<think>\n\n</think>\n\nps shows information ab...,None,
6,What does kill do in Linux?,<think>\n\n</think>\n\nkill sends a signal to ...,None,
7,What is fork followed by exec used for?,<think>\n\n</think>\n\nFork creates a child pr...,None,
8,What is a shell pipeline?,<think>\n\n</think>\n\nA shell pipeline connec...,None,
9,What is the root directory in Linux?,<think>\n\n</think>\n\nThe root directory is t...,None,


In [ ]:
scores = [
    2,  # 1  Shell
    2,  # 2  pwd
    2,  # 3  ls
    2,  # 4  cd
    2,  # 5  chmod
    2,  # 6  ps
    2,  # 7  kill
    2,  # 8  fork + exec
    2,  # 9  shell pipeline
    2,  # 10 root directory
    2,  # 11 protection
    2,  # 12 authentication
    2,  # 13 authorization
    2,  # 14 access-control list
    2,  # 15 least privilege
    2,  # 16 privilege escalation
    2,  # 17 process isolation
    2,  # 18 secure boot
    1,  # 19 OS vulnerability - correct core idea, less complete
    2,  # 20 software updates
    2,  # 21 distributed OS
    2,  # 22 real-time deadline
    2,  # 23 hard real-time
    2,  # 24 soft real-time
    2,  # 25 SMP
    2,  # 26 processor affinity
    2,  # 27 load balancing
    2,  # 28 NUMA
    2,  # 29 embedded OS
    2   # 30 OS API
]

reasons = [
    "Correct definition of a shell.",
    "Correctly explains pwd.",
    "Correctly explains ls.",
    "Correctly explains cd.",
    "Correctly explains chmod and permissions.",
    "Correctly explains ps and process information.",
    "Correctly explains that kill sends a signal to a process.",
    "Correctly explains fork followed by exec.",
    "Correctly explains a shell pipeline.",
    "Correctly identifies / as the root directory.",
    "Correctly explains OS protection.",
    "Correctly explains authentication.",
    "Correctly explains authorization.",
    "Correctly explains an access-control list.",
    "Correctly explains least privilege.",
    "Correctly explains privilege escalation.",
    "Correctly explains the purpose of process isolation.",
    "Correctly explains the purpose of secure boot.",
    "Core idea is correct, but the answer is less complete than the expected answer.",
    "Correctly explains the security importance of updates.",
    "Correctly explains a distributed operating system.",
    "Correctly explains a real-time deadline.",
    "Correctly explains hard real-time scheduling.",
    "Correctly explains soft real-time scheduling.",
    "Correctly explains symmetric multiprocessing.",
    "Correctly explains processor affinity.",
    "Correctly explains load balancing.",
    "Correctly explains NUMA.",
    "Correctly explains an embedded operating system.",
    "Correctly explains an OS API."
]

evaluation_df["Semantic Score (0-2)"] = scores
evaluation_df["Reason"] = reasons

total_score = sum(scores)
max_score = len(scores) * 2
semantic_score = (total_score / max_score) * 100

print("===================================")
print("SEMANTIC EVALUATION")
print("===================================")
print(f"Test questions: {len(scores)}")
print(f"Total score: {total_score}/{max_score}")
print(f"Semantic Score: {semantic_score:.2f}%")
print("===================================")

display(
    evaluation_df[
        ["Question", "Semantic Score (0-2)", "Reason"]
    ]
)

SEMANTIC EVALUATION
Test questions: 30
Total score: 59/60
Semantic Score: 98.33%


,Question,Semantic Score (0-2),Reason
0,What is a shell in Linux?,2,Correct definition of a shell.
1,What does the Linux command pwd show?,2,Correctly explains pwd.
2,What does the Linux command ls do?,2,Correctly explains ls.
3,What does the Linux command cd do?,2,Correctly explains cd.
4,What does chmod do in Linux?,2,Correctly explains chmod and permissions.
5,What does ps show in Linux?,2,Correctly explains ps and process information.
6,What does kill do in Linux?,2,Correctly explains that kill sends a signal to...
7,What is fork followed by exec used for?,2,Correctly explains fork followed by exec.
8,What is a shell pipeline?,2,Correctly explains a shell pipeline.
9,What is the root directory in Linux?,2,Correctly identifies / as the root directory.


In [ ]:
# Final evaluation statistics

score_counts = evaluation_df["Semantic Score (0-2)"].value_counts().sort_index()

total_score = evaluation_df["Semantic Score (0-2)"].sum()
max_score = len(evaluation_df) * 2
semantic_score = (total_score / max_score) * 100

print("======================================")
print("     OSTutorLLM FINAL EVALUATION")
print("======================================")
print(f"Test questions        : {len(evaluation_df)}")
print(f"Score 2 (Fully correct): {score_counts.get(2, 0)}")
print(f"Score 1 (Partially correct): {score_counts.get(1, 0)}")
print(f"Score 0 (Incorrect)   : {score_counts.get(0, 0)}")
print(f"Total score           : {total_score}/{max_score}")
print(f"Semantic score        : {semantic_score:.2f}%")
print("======================================")

# Save complete evaluation
FINAL_EVAL_FILE = (
    "/content/drive/MyDrive/OSTutorLLM/"
    "ostutorllm_final_30_question_evaluation.csv"
)

evaluation_df.to_csv(FINAL_EVAL_FILE, index=False)

print("\nEvaluation saved to:")
print(FINAL_EVAL_FILE)

     OSTutorLLM FINAL EVALUATION
Test questions        : 30
Score 2 (Fully correct): 29
Score 1 (Partially correct): 1
Score 0 (Incorrect)   : 0
Total score           : 59/60
Semantic score        : 98.33%

Evaluation saved to:
/content/drive/MyDrive/OSTutorLLM/ostutorllm_final_30_question_evaluation.csv
